In [0]:
%sql
USE CATALOG `retail-sales-data-warehouse`;
USE SCHEMA silver;

In [0]:
%sql
CREATE OR REPLACE TABLE stores_silver
USING DELTA
AS

SELECT
    store_id,
    store_name,
    region,
    silver_load_time
FROM (

    SELECT
        CAST(StoreID AS INT)                   AS store_id,
        INITCAP(TRIM(StoreName))               AS store_name,
        TRIM(Region)                           AS region,
        CURRENT_TIMESTAMP()                    AS silver_load_time,

        ROW_NUMBER() OVER (
            PARTITION BY StoreID
            ORDER BY CURRENT_TIMESTAMP() DESC
        ) AS rn

    FROM bronze.stores_raw
    WHERE StoreID IS NOT NULL

)

WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE sales_silver
USING DELTA
AS

SELECT
    transaction_id,
    customer_id,
    product_id,
    store_id,
    quantity,
    transaction_date,
    silver_load_time
FROM (

    SELECT
        CAST(TransactionID AS INT)                 AS transaction_id,
        CAST(CustomerID AS INT)                    AS customer_id,
        CAST(ProductID AS INT)                     AS product_id,
        CAST(StoreID AS INT)                       AS store_id,
        CAST(Quantity AS INT)                      AS quantity,

        COALESCE(
            TRY_TO_DATE(TxnDate, 'd-M-yyyy'),
            TRY_TO_DATE(TxnDate, 'M/d/yyyy')
        )                                          AS transaction_date,

        CURRENT_TIMESTAMP()                        AS silver_load_time,

        ROW_NUMBER() OVER (
            PARTITION BY TransactionID
            ORDER BY CURRENT_TIMESTAMP() DESC
        ) AS rn

    FROM bronze.sales_raw
    WHERE TransactionID IS NOT NULL
    AND Quantity > 0

)

WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE customers_silver
USING DELTA
AS

SELECT
    customer_id,
    customer_name,
    email,
    city,
    address,
    silver_load_time
FROM (

    SELECT
        CAST(CustomerID AS INT)                  AS customer_id,
        INITCAP(TRIM(CustomerName))              AS customer_name,
        LOWER(TRIM(Email))                       AS email,
        INITCAP(TRIM(City))                      AS city,
        TRIM(Address)                            AS address,
        CURRENT_TIMESTAMP()                      AS silver_load_time,

        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY CURRENT_TIMESTAMP() DESC
        ) AS rn

    FROM bronze.customers_raw
    WHERE CustomerID IS NOT NULL

)

WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE products_silver
USING DELTA
AS

SELECT
    product_id,
    product_name,
    category,
    unit_price,
    silver_load_time
FROM (

    SELECT
        CAST(ProductID AS INT)                  AS product_id,
        INITCAP(TRIM(ProductName))              AS product_name,
        INITCAP(TRIM(Category))                 AS category,
        CAST(UnitPrice AS DECIMAL(10,2))        AS unit_price,
        CURRENT_TIMESTAMP()                     AS silver_load_time,

        ROW_NUMBER() OVER (
            PARTITION BY ProductID
            ORDER BY CURRENT_TIMESTAMP() DESC
        ) AS rn

    FROM bronze.products_raw
    WHERE ProductID IS NOT NULL
    AND UnitPrice > 0

)

WHERE rn = 1;

In [0]:
%sql
SELECT 'customers_silver' AS table_name, COUNT(*) FROM customers_silver
UNION ALL
SELECT 'products_silver', COUNT(*) FROM products_silver
UNION ALL
SELECT 'stores_silver', COUNT(*) FROM stores_silver
UNION ALL
SELECT 'sales_silver', COUNT(*) FROM sales_silver;